In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv(
    "/Users/gabrielaslomiany/GIT/Alzheimer Detection/data/data.csv",
    sep=None,
    engine="python"
)
print(data.shape)
data.head()

(1737, 361)


,﻿RID,Test_data,Age,Gender,Year_education,Ethnicity,Race,Marital_status,High_risk_ApoE4,Ventricles,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,0,"74,3",Male,16,Not Hisp/Latino,White,Married,0.0,118233.0,...,"0,694",3835,1622,"2,077","0,746",15683,4272,"3,028","0,649",4396
1,3,0,"81,3",Male,18,Not Hisp/Latino,White,Married,1.0,84599.0,...,"0,591",3681,1734,"1,942","0,696",10387,3316,"2,545","0,686",3304
2,4,1,"67,5",Male,10,Hisp/Latino,White,Married,0.0,39605.0,...,"0,588",4060,1728,"2,18","0,607",11156,3598,"2,67","0,631",1338
3,5,0,"73,7",Male,16,Not Hisp/Latino,White,Married,0.0,34062.0,...,"0,628",5180,1868,"2,543","0,709",11579,3387,"2,911","0,66",1623
4,6,0,"80,4",Female,13,Not Hisp/Latino,White,Married,0.0,39826.0,...,"0,631",3078,1241,"2,141","0,701",9641,2781,"2,9","0,727",1035


In [3]:
cognitive = pd.read_csv(
    "/Users/gabrielaslomiany/GIT/Alzheimer Detection/data/cognitive_score.csv",
    sep=None,
    engine="python"
)
print(cognitive.shape)
cognitive.head()

(1737, 10)


,﻿RID,Test_data,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,RAVLT_perc_forgetting
0,2,0,0,"10,67","18,67",28,44.0,4.0,6.0,"54,5455"
1,3,0,"4,5",22,31,20,22.0,1.0,4.0,100
2,4,1,1,"14,33","21,33",27,37.0,7.0,4.0,"36,3636"
3,5,0,0,"8,67","14,67",29,37.0,4.0,4.0,"44,4444"
4,6,0,"0,5","18,67","25,67",25,30.0,1.0,5.0,"83,3333"


In [4]:
diagnosis = pd.read_csv(
    "/Users/gabrielaslomiany/GIT/Alzheimer Detection/data/diagnosis_target.csv",
    sep=None,
    engine="python"
)
print(diagnosis.shape)
diagnosis.head()

(1737, 4)


,﻿RID,Test_data,Diagnosis,FDG_PET
0,2,0,CN,"1,36926"
1,3,0,AD,"1,09079"
2,4,1,LMCI,NaN
3,5,0,CN,"1,29799"
4,6,0,LMCI,NaN


### Missing data

##### Data datframe

In [5]:
data.isna().sum()

﻿RID                                                            0
Test_data                                                       0
Age                                                             0
Gender                                                          0
Year_education                                                  0
                                                               ..
Volume (Cortical Parcellation) of RightMiddleTemporal           0
Surface Area of RightMiddleTemporal                             0
Cortical Thickness Average of RightMiddleTemporal               0
Cortical Thickness Standard Deviation of RightMiddleTemporal    0
Volume (WM Parcellation) of FourthVentricle                     0
Length: 361, dtype: int64

As data dataframe has 361 columns, checking NaN's is much simpler with function below. Data's share is (1737, 361), and count highest count of missing values is 291, which is about 17%.

In [6]:
missing_values = data.isna().sum()
missing_values = missing_values[missing_values > 0]

missing_values

High_risk_ApoE4                                                  12
Ventricles                                                       82
Hippocampus                                                     248
WholeBrain                                                       48
Entorhinal                                                      271
Fusiform                                                        271
MidTemp                                                         271
Intra cranial volume                                             15
Volume (WM Parcellation) of RightVessel                           2
Volume (Cortical Parcellation) of LeftEntorhinal                  1
Surface Area of LeftEntorhinal                                    1
Cortical Thickness Average of LeftEntorhinal                      1
Cortical Thickness Standard Deviation of LeftEntorhinal           1
Volume (Cortical Parcellation) of LeftParahippocampal             1
Surface Area of LeftParahippocampal             

In [7]:
data[['Hippocampus','Entorhinal', 'Fusiform', 'MidTemp', 'Volume (WM Parcellation) of FifthVentricle']].dtypes


Hippocampus                                   float64
Entorhinal                                    float64
Fusiform                                      float64
MidTemp                                       float64
Volume (WM Parcellation) of FifthVentricle        str
dtype: object

Hippocampus, Entorhinal, Fusiform, MidTemp --> replace with mean, as their dtype is float
Volume (WM Parcellation) of FifthVentricle --> drop

In [8]:
columns = ["Entorhinal", "Fusiform", "MidTemp"]

data[columns] = data[columns].fillna(data[columns].mean())

In [9]:
data.dropna(inplace=True)

In [10]:
data.isna().sum()

﻿RID                                                            0
Test_data                                                       0
Age                                                             0
Gender                                                          0
Year_education                                                  0
                                                               ..
Volume (Cortical Parcellation) of RightMiddleTemporal           0
Surface Area of RightMiddleTemporal                             0
Cortical Thickness Average of RightMiddleTemporal               0
Cortical Thickness Standard Deviation of RightMiddleTemporal    0
Volume (WM Parcellation) of FourthVentricle                     0
Length: 361, dtype: int64

##### Cognitive dataframe

In [11]:
cognitive.isna().sum()

﻿RID                      0
Test_data                 0
CDRSB                     0
ADAS11                    5
ADAS13                   14
MMSE                      0
RAVLT_immediate           6
RAVLT_learning            6
RAVLT_forgetting          6
RAVLT_perc_forgetting    11
dtype: int64

Cognitive dataframe shape is (1737, 10), since missing values are less than 5% I will drop it

In [12]:
cognitive.dropna(inplace=True)

In [13]:
cognitive.isna().sum().sum()

np.int64(0)

##### Diagnosis dataframe

In [14]:
print(diagnosis.shape)
diagnosis.isna().sum()

(1737, 4)


﻿RID           0
Test_data      0
Diagnosis      0
FDG_PET      434
dtype: int64

In [15]:
diagnosis.dtypes

﻿RID         int64
Test_data    int64
Diagnosis      str
FDG_PET        str
dtype: object

In [17]:
# Even though NANs are big portion of FDG_PET after numerous attempts I decided to drop it

diagnosis.dropna()

,﻿RID,Test_data,Diagnosis,FDG_PET
0,2,0,CN,"1,36926"
1,3,0,AD,"1,09079"
3,5,0,CN,"1,29799"
6,8,0,CN,"1,27628"
7,10,0,AD,"1,11881"
...,...,...,...,...
1730,5275,0,AD,"1,03993"
1731,5282,0,SMC,"1,13549"
1732,5287,0,SMC,"1,58312"
1733,5295,0,SMC,"1,16317"


##### Merging dataframes

In [ ]:
merged = pd.merge(diagnosis, cognitive, on="Test_data", how="inner")
df = pd.merge(merged, data, on="Test_data", how="inner")
df.head()

In [22]:
print(diagnosis.columns.tolist())

['\ufeffRID', 'Test_data', 'Diagnosis', 'FDG_PET']


In [23]:
print(cognitive.columns.tolist())

['\ufeffRID', 'Test_data', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting']


In [25]:
print(data.columns.tolist())

['\ufeffRID', 'Test_data', 'Age', 'Gender', 'Year_education', 'Ethnicity', 'Race', 'Marital_status', 'High_risk_ApoE4', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'Intra cranial volume', 'Volume (WM Parcellation) of RightPallidum', 'Volume (Cortical Parcellation) of RightParacentral', 'Surface Area of RightParacentral', 'Cortical Thickness Average of RightParacentral', 'Cortical Thickness Standard Deviation of RightParacentral', 'Volume (Cortical Parcellation) of RightParahippocampal', 'Surface Area of RightParahippocampal', 'Cortical Thickness Average of RightParahippocampal', 'Cortical Thickness Standard Deviation of RightParahippocampal', 'Volume (Cortical Parcellation) of RightParsOpercularis', 'Surface Area of RightParsOpercularis', 'Cortical Thickness Average of RightParsOpercularis', 'Cortical Thickness Standard Deviation of RightParsOpercularis', 'Volume (Cortical Parcellation) of RightParsOrbitalis', 'Surface Area of RightParsOrbitalis', 'C

##### Standardise numerical columns

In [ ]:
num_col = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
scaler = StandardScaler()
df['num_col'] = scaler.fit_transform(df[num_col])